In [1]:
import pandas as pd
from sklift.datasets import fetch_hillstrom

# Fetching all targets so we keep our guardrail metrics
bunch = fetch_hillstrom(target_col='all')

# Reassemble
df = bunch.data.copy()
df['treatment'] = bunch.treatment
df['visit'] = bunch.target['visit']
df['conversion'] = bunch.target['conversion']
df['spend'] = bunch.target['spend']

print('-'*50)
print(df.shape)
print('-'*50)
print(df.head())
print('-'*50)
print(df['treatment'].value_counts())
print('-'*50)
print(df['visit'].value_counts())
print('-'*50)

--------------------------------------------------
(64000, 12)
--------------------------------------------------
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

       treatment  visit  conversion  spend  
0  Womens E-Mail      0           0    0.0  
1      No E-Mail      0           0    0.0  
2  Womens E-Mail      0           0    0.0  
3    Mens E-Mail      0           0    0.0  
4  Womens E-Mail      0           0    0.0  
--------------------------------------------------
treatment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name:

In [2]:
import sqlite3

conn = sqlite3.connect('../data/hillstrom.db')

df.to_sql('raw_customers', conn, if_exists='replace', index=False)

print('-'*50)
print(pd.read_sql(
    'SELECT * FROM raw_customers LIMIT 5',
    conn
))
print('-'*50)
print(pd.read_sql(
    'SELECT COUNT(*) AS row_count FROM raw_customers',
    conn
))
print('-'*50)

--------------------------------------------------
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

       treatment  visit  conversion  spend  
0  Womens E-Mail      0           0    0.0  
1      No E-Mail      0           0    0.0  
2  Womens E-Mail      0           0    0.0  
3    Mens E-Mail      0           0    0.0  
4  Womens E-Mail      0           0    0.0  
--------------------------------------------------
   row_count
0      64000
--------------------------------------------------


In [3]:
df.to_csv('../data/raw/hillstrom_raw.csv', index=False)

conn.execute("DROP TABLE IF EXISTS experiment")
conn.execute("""
    CREATE TABLE experiment AS
    SELECT * FROM raw_customers
    WHERE treatment != 'Mens E-Mail';
""")

conn.commit()

df_exp = pd.read_sql("SELECT * FROM experiment", conn)
print(f"Row count after dropping Mens E-Mail: {len(df_exp)}")

pd.read_sql(
    "SELECT treatment, COUNT(*) AS n FROM experiment GROUP BY treatment",
    conn
)

Row count after dropping Mens E-Mail: 42693


,treatment,n
0,No E-Mail,21306
1,Womens E-Mail,21387


In [4]:
from scipy.stats import chisquare

counts = df_exp['treatment'].value_counts()

stat, pval = chisquare(counts, f_exp=[counts.sum()/2, counts.sum()/2])
print(f"Chi-square p-value: {pval:.4f}")

Chi-square p-value: 0.6950


The SRM chi-square test produced a p-value of 0.6950,
so we did not find evidence of a mismatch from the expected 50/50 treatment allocation.